# Data Formulator + local model (Ollama) on Google Colab

Runs the whole stack on one Colab GPU runtime: Ollama serving a local model, and
Data Formulator configured to use it through the native `ollama_chat` endpoint
(the same setup as `.env.template`).

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better), then `Runtime > Run all`.
The first run takes roughly 10-15 minutes (model download + frontend build).

If the repository is private, add a GitHub token as a Colab secret named `GH_TOKEN`
(key icon in the left sidebar) and grant this notebook access to it.

In [ ]:
#@title 1. Settings
REPO = "shervinghaffari79/Analytical_Intelligence_System"
BRANCH = "claude/tender-ramanujan-v6xxbu"  #@param {type:"string"}
MODEL = "qwen3.5:9b"  #@param {type:"string"}
NUM_CTX = 32768  #@param {type:"integer"}
PORT = 5567

import os, subprocess, time, urllib.request
APP_DIR = "/content/Analytical_Intelligence_System"

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0:
    print("GPU:", gpu.stdout.strip())
else:
    print("WARNING: no GPU runtime. The model will run on CPU and be very slow.\n"
          "Switch via Runtime > Change runtime type > T4 GPU, then run all again.")

In [ ]:
#@title 2. Get the code
if not os.path.isdir(APP_DIR):
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN") or ""
    except Exception:
        pass  # no secret set: fine for a public repository
    auth = f"x-access-token:{token}@" if token else ""
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        f"https://{auth}github.com/{REPO}.git", APP_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("Clone failed. If the repository is private, add a GH_TOKEN "
                         "Colab secret (see the top of this notebook).\n"
                         + (r.stderr.replace(token, "***") if token else r.stderr))
os.chdir(APP_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
#@title 3. Install Ollama, start it, pull the model
!command -v zstd >/dev/null || (apt-get -qq update && apt-get -qq install -y zstd pciutils >/dev/null)
!command -v ollama >/dev/null || (curl -fsSL https://ollama.com/install.sh | sh)

def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        return True
    except Exception:
        return False

if not ollama_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
    else:
        raise SystemExit("Ollama did not start; see /content/ollama.log")

!ollama pull {MODEL}
!ollama list

In [ ]:
#@title 4. Install Data Formulator (backend deps + frontend build)
%cd {APP_DIR}
!command -v uv >/dev/null || pip -q install uv
!uv sync -q

# Vite 7 needs Node >= 20.19; upgrade if the runtime ships something older.
node_ok = subprocess.run(
    ["node", "-e", "const [a,b]=process.versions.node.split('.').map(Number);"
                   "process.exit(a>22||(a===22&&b>=12)||(a===20&&b>=19)?0:1)"]).returncode == 0
if not node_ok:
    !npm install -g -s n && n -q 22 && hash -r
!command -v yarn >/dev/null || npm install -g -s yarn
!node -v
!yarn install --frozen-lockfile --silent
!yarn build 2>&1 | tail -n 5
assert os.path.exists("py-src/data_formulator/dist/index.html"), "frontend build failed"

In [ ]:
#@title 5. Configure Data Formulator for the local model
env = f"""SANDBOX=local
LOCAL_ENABLED=true
LOCAL_ENDPOINT=ollama_chat
LOCAL_API_BASE=http://localhost:11434
LOCAL_API_KEY=ollama
LOCAL_MODELS={MODEL}
DF_OLLAMA_NUM_CTX={NUM_CTX}
"""
open(os.path.join(APP_DIR, ".env"), "w").write(env)
print(env)

In [ ]:
#@title 6. Smoke-test the model (content, tool calls, context window)
!uv run python tools/probe_local_model.py --model {MODEL} 2>&1 | tail -n 60

In [ ]:
#@title 7. Start Data Formulator and open it
def app_up():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
        return True
    except Exception:
        return False

if not app_up():
    subprocess.Popen(["uv", "run", "data_formulator", "--port", str(PORT), "--dev"],
                     cwd=APP_DIR, stdout=open("/content/data_formulator.log", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(120):
        if app_up():
            break
        time.sleep(1)
    else:
        !tail -n 40 /content/data_formulator.log
        raise SystemExit("Data Formulator did not start; log above")

from google.colab import output
print("Data Formulator is running. Open it in a new tab:")
output.serve_kernel_port_as_window(PORT, anchor_text="Open Data Formulator (workspace)")
output.serve_kernel_port_as_window(PORT, path="/chat", anchor_text="Open the conversational UI (/chat)")

In the app, pick the `local` model in the model selector. The link above only works
for your own Google account while this runtime is alive.

**Troubleshooting**

- Logs: run `!tail -n 50 /content/data_formulator.log` or `!tail -n 50 /content/ollama.log`.
- GPU memory: `!ollama ps` shows whether the model is fully on the GPU. If it is
  split with CPU, lower `NUM_CTX` in cell 1 (for example 16384), rerun cell 5, then restart the app with the cell below.
- If streaming responses stall behind Colab's port proxy, use the optional tunnel cell at the bottom.

In [ ]:
#@title Restart Data Formulator (after changing settings)
# Tick the box and run this cell on its own; it is skipped during "Run all".
RESTART = False  #@param {type:"boolean"}
if RESTART:
    !pkill -f "data_formulator --port" ; sleep 2
    print("Stopped. Now rerun cell 7.")

In [ ]:
#@title Optional: public tunnel instead of the Colab proxy
# WARNING: this publishes the app on a public URL with no login. Anyone with the
# link can use it and run analysis code on this runtime. Only run this if the
# Colab link above does not work, and stop the runtime when you are done.
ENABLE_PUBLIC_TUNNEL = False  #@param {type:"boolean"}
if ENABLE_PUBLIC_TUNNEL:
    !test -x /content/cloudflared || (wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared)
    subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                     stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT,
                     start_new_session=True)
    time.sleep(8)
    !grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /content/tunnel.log | head -n 1